In [18]:
import pandas as pd
import sqlite3
import os

db_file = "test_data.db"
conn = sqlite3.connect(os.path.join(os.getcwd(), "notebooks", "example", db_file))

In [19]:
df = pd.read_csv(os.path.join(os.getcwd(), "notebooks", "example", "test_data.tsv"), sep="\t", index_col="ID")
df

,Gene Symbols,Synonyms
ID,,
BLA:1,A,"A1,A2,A3"
BLA:2,B,"B1,B2"
BLA:3,C,"C1,C2,C3,C4"


In [20]:
df.index = df.index.str.split(":").str[1].astype(int)
df

,Gene Symbols,Synonyms
ID,,
1,A,"A1,A2,A3"
2,B,"B1,B2"
3,C,"C1,C2,C3,C4"


In [21]:
df.index.rename("id", inplace=True)
df

,Gene Symbols,Synonyms
id,,
1,A,"A1,A2,A3"
2,B,"B1,B2"
3,C,"C1,C2,C3,C4"


In [22]:
df.columns = df.columns.str.lower()
df

,gene symbols,synonyms
id,,
1,A,"A1,A2,A3"
2,B,"B1,B2"
3,C,"C1,C2,C3,C4"


In [23]:
df.rename(columns={"gene symbols": "gene_symbols"}, inplace=True)
df

,gene_symbols,synonyms
id,,
1,A,"A1,A2,A3"
2,B,"B1,B2"
3,C,"C1,C2,C3,C4"


In [24]:
df["synonym"] = df["synonyms"].str.split(",")
df

,gene_symbols,synonyms,synonym
id,,,
1,A,"A1,A2,A3","[A1, A2, A3]"
2,B,"B1,B2","[B1, B2]"
3,C,"C1,C2,C3,C4","[C1, C2, C3, C4]"


In [25]:
df_synonym = df.synonym.explode()
df_synonym

id
1    A1
1    A2
1    A3
2    B1
2    B2
3    C1
3    C2
3    C3
3    C4
Name: synonym, dtype: str

In [26]:
df_synonym = df_synonym.to_frame()
df_synonym

,synonym
id,
1,A1
1,A2
1,A3
2,B1
2,B2
3,C1
3,C2
3,C3
3,C4


In [27]:
df_synonym["gene_id"] = df_synonym.index
df_synonym

,synonym,gene_id
id,,
1,A1,1
1,A2,1
1,A3,1
2,B1,2
2,B2,2
3,C1,3
3,C2,3
3,C3,3
3,C4,3


In [28]:
df_synonym.reset_index(drop=True, inplace=True)
df_synonym

,synonym,gene_id
0,A1,1
1,A2,1
2,A3,1
3,B1,2
4,B2,2
5,C1,3
6,C2,3
7,C3,3
8,C4,3


In [29]:
df_synonym.index += 1
df_synonym

,synonym,gene_id
1,A1,1
2,A2,1
3,A3,1
4,B1,2
5,B2,2
6,C1,3
7,C2,3
8,C3,3
9,C4,3


In [30]:
df_synonym.index.rename("id", inplace=True)
df_synonym

,synonym,gene_id
id,,
1,A1,1
2,A2,1
3,A3,1
4,B1,2
5,B2,2
6,C1,3
7,C2,3
8,C3,3
9,C4,3


In [31]:
df_synonym.to_sql("synonym", conn, index_label="id", if_exists="replace")

9

In [32]:
df_gene = df[["gene_symbols"]].copy()
df_gene.rename(columns={"gene_symbols": "symbol"}, inplace=True)
df_gene


,symbol
id,
1,A
2,B
3,C


In [33]:
df_gene.to_sql("gene", conn, index_label="id", if_exists="replace")

3

In [ ]:
sql = """
    SELECT 
        g.symbol, s.synonym
    FROM 
        gene AS g INNER JOIN synonym AS s 
        ON (g.id = s.gene_id)
    WHERE s.synonym LIKE 'A%'
    GROUP BY g.symbol, s.synonym
"""

pd.read_sql(sql, conn)

,symbol,synonym
0,A,A1
1,A,A2
2,A,A3
3,B,B1
4,B,B2
5,C,C1
6,C,C2
7,C,C3
8,C,C4
